# ***TRIDENT*** (<font color="red">T</font>emporal <font color="red">R</font>einforcement <font color="red">I</font>ntrusion <font color="red">D</font>etection <font color="red">E</font>ngine for <font color="red">N</font>etwork <font color="red">T</font>raffic)


In [ ]:
import numpy as np
import pandas as pd

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


/kaggle/input/cicddos2019/Syn-training.parquet
/kaggle/input/cicddos2019/UDPLag-testing.parquet
/kaggle/input/cicddos2019/NetBIOS-testing.parquet
/kaggle/input/cicddos2019/Portmap-training.parquet
/kaggle/input/cicddos2019/Syn-testing.parquet
/kaggle/input/cicddos2019/MSSQL-testing.parquet
/kaggle/input/cicddos2019/SNMP-testing.parquet
/kaggle/input/cicddos2019/UDPLag-training.parquet
/kaggle/input/cicddos2019/NTP-testing.parquet
/kaggle/input/cicddos2019/LDAP-testing.parquet
/kaggle/input/cicddos2019/UDP-training.parquet
/kaggle/input/cicddos2019/NetBIOS-training.parquet
/kaggle/input/cicddos2019/DNS-testing.parquet
/kaggle/input/cicddos2019/UDP-testing.parquet
/kaggle/input/cicddos2019/LDAP-training.parquet
/kaggle/input/cicddos2019/TFTP-testing.parquet
/kaggle/input/cicddos2019/MSSQL-training.parquet


# ==========================================
# 1. SETUP & CONFIGURATION
# ==========================================

In [ ]:
import os
import gc
import numpy as np
import pandas as pd
import glob
import time
import psutil
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, losses, regularizers
from collections import deque
import random


try:
    from tensorflow.keras import mixed_precision
    policy = mixed_precision.Policy('mixed_float16')
    mixed_precision.set_global_policy(policy)
    print("⚡ Mixed Precision (FP16) Enabled for P100 GPU")
except Exception as e:
    print(f"⚠️ Mixed Precision setup failed: {e}")


CONFIG = {
    'SAMPLE_RATIO': 0.10,
    'SEED': 42,
    'BATCH_SIZE': 128,
    'EPOCHS': 15,
    'RFE_STEPS': 15,
    'TARGET_FEATURES': 20,
    'GAMMA': 0.01,
    'LEARNING_RATE': 0.001,
    'EPSILON_DECAY': 0.995,
    'MIN_EPSILON': 0.01,
    'L2_REG': 0.001,
    'PATIENCE': 8
}


gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        tf.config.experimental.set_memory_growth(gpus[0], True)
        print(f"✅ GPU Detected: {tf.config.experimental.get_device_details(gpus[0])['device_name']}")
    except RuntimeError as e:
        print(e)
else:
    print("⚠️ Running on CPU - This will be slow!")

def print_memory():
    print(f"RAM Used: {psutil.virtual_memory().percent}%")

2026-01-21 03:27:32.813637: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768966053.016652      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768966053.084482      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768966053.600929      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768966053.600983      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768966053.600986      55 computation_placer.cc:177] computation placer alr

⚡ Mixed Precision (FP16) Enabled for P100 GPU
✅ GPU Detected: Tesla P100-PCIE-16GB


# ==========================================
# 2. DATA LOADING & OPTIMIZATION
# ==========================================


In [ ]:
def load_cicddos_data(base_path, sample_ratio=0.1):
    """
    Loads and samples from multiple parquet files efficiently.
    """
    all_files = glob.glob(os.path.join(base_path, "*.parquet"))
    print(f"Found {len(all_files)} parquet files.")

    df_list = []

    for f in all_files:
        try:

            temp_df = pd.read_parquet(f)


            if sample_ratio < 1.0:
                temp_df = temp_df.sample(frac=sample_ratio, random_state=CONFIG['SEED'])

            df_list.append(temp_df)

            del temp_df
            gc.collect()

        except Exception as e:
            print(f"Error loading {f}: {e}")

    if not df_list:
        raise ValueError("No data loaded!")

    full_df = pd.concat(df_list, axis=0, ignore_index=True)


    full_df.replace([np.inf, -np.inf], np.nan, inplace=True)
    full_df.dropna(inplace=True)

    print(f"\n✅ Total Dataset Shape: {full_df.shape}")
    print_memory()
    return full_df


BASE_PATH = "/kaggle/input/cicddos2019"
df = load_cicddos_data(BASE_PATH, sample_ratio=CONFIG['SAMPLE_RATIO'])


df['Label_Binary'] = df['Label'].apply(lambda x: 0 if str(x).lower() == 'benign' else 1)
y = df['Label_Binary'].values
X_raw = df.drop(['Label', 'Label_Binary'], axis=1)


X_raw.columns = X_raw.columns.str.strip()
print(f"Class Distribution:\n{df['Label_Binary'].value_counts(normalize=True)}")

Found 17 parquet files.

✅ Total Dataset Shape: (43135, 78)
RAM Used: 5.9%
Class Distribution:
Label_Binary
1    0.778046
0    0.221954
Name: proportion, dtype: float64


# ==========================================
# 3. STAGE 1: FEATURE ENGINEERING (88 -> ~128)
# ==========================================

In [ ]:
def enhanced_feature_engineering(X):

    print("\n🚀 Starting Stage 1: Feature Engineering...")
    X_eng = X.copy()


    if 'Flow IAT Mean' in X_eng.columns and 'Flow IAT Std' in X_eng.columns:

        X_eng['IAT_Variance_Ratio'] = X_eng['Flow IAT Std'] / (X_eng['Flow IAT Mean'] + 1e-6)

        X_eng['IAT_Entropy_Approx'] = np.log1p(X_eng['Flow IAT Std'])


    if 'Fwd Pkt Len Std' in X_eng.columns:
        X_eng['Wavelet_Energy_L1_Proxy'] = X_eng['Fwd Pkt Len Std'] ** 2
        X_eng['Wavelet_Energy_L2_Proxy'] = np.sqrt(X_eng['Fwd Pkt Len Std'])


    if 'Flow Pkts/s' in X_eng.columns:
        X_eng['Dominant_Freq_Proxy'] = X_eng['Flow Pkts/s']


    if 'Protocol' in X_eng.columns:
        X_eng['Protocol_Complexity'] = X_eng['Protocol'] * np.log1p(X_eng['Flow Duration'])


    if 'Fwd Header Len' in X_eng.columns and 'Bwd Header Len' in X_eng.columns:
        X_eng['Direction_Asymmetry'] = np.abs(X_eng['Fwd Header Len'] - X_eng['Bwd Header Len']) / (X_eng['Fwd Header Len'] + X_eng['Bwd Header Len'] + 1e-6)


    X_eng.replace([np.inf, -np.inf], 0, inplace=True)
    X_eng.fillna(0, inplace=True)

    print(f"Feature count expanded: {X.shape[1]} -> {X_eng.shape[1]}")
    return X_eng

X_eng = enhanced_feature_engineering(X_raw)


scaler = MinMaxScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X_eng), columns=X_eng.columns)

del X_raw
gc.collect()


🚀 Starting Stage 1: Feature Engineering...
Feature count expanded: 77 -> 80


0

# ==========================================
# 4. STAGE 2.1: RF PRE-FILTERING (128 -> 35)
# ==========================================

In [ ]:
print("\n🌲 Starting Stage 2.1: Random Forest Pre-filtering...")

rf = RandomForestClassifier(n_estimators=50, n_jobs=-1, random_state=CONFIG['SEED'])
rf.fit(X_scaled, y)

importances = rf.feature_importances_
indices = np.argsort(importances)[::-1]

TOP_K_RF = 35
top_35_indices = indices[:TOP_K_RF]
top_35_features = X_scaled.columns[top_35_indices]

print(f"✅ Selected Top {TOP_K_RF} Features:")
print(list(top_35_features[:5]), "...")

X_rf_selected = X_scaled[top_35_features]


🌲 Starting Stage 2.1: Random Forest Pre-filtering...
✅ Selected Top 35 Features:
['Bwd Packets/s', 'Packet Length Min', 'Avg Packet Size', 'Down/Up Ratio', 'Bwd Packet Length Mean'] ...


# ==========================================
# 5. STAGE 2.2: TCN-BASED RFE (35 -> 20)
# ==========================================

In [ ]:
print("\n🧠 Starting Stage 2.2: TCN-based RFE (Dilated Causal Convolutions)...")

def build_rfe_tcn(input_dim):
    """
    TCN Architecture with L2 Regularization
    """
    inputs = layers.Input(shape=(input_dim, 1))


    def residual_block(x, dilation_rate):
        shortcut = x


        x = layers.Conv1D(64, kernel_size=3, dilation_rate=dilation_rate,
                          padding='causal', activation='linear',
                          kernel_regularizer=regularizers.l2(CONFIG['L2_REG']))(x)
        x = layers.LayerNormalization()(x)
        x = layers.Activation('relu')(x)
        x = layers.SpatialDropout1D(0.5)(x)


        x = layers.Conv1D(64, kernel_size=3, dilation_rate=dilation_rate,
                          padding='causal', activation='linear',
                          kernel_regularizer=regularizers.l2(CONFIG['L2_REG']))(x)
        x = layers.LayerNormalization()(x)
        x = layers.Activation('relu')(x)
        x = layers.SpatialDropout1D(0.5)(x)

        if shortcut.shape[-1] != 64:
             shortcut = layers.Conv1D(64, kernel_size=1, padding='same')(shortcut)

        x = layers.Add()([x, shortcut])
        return x

    x = inputs
    x = residual_block(x, dilation_rate=1)
    x = residual_block(x, dilation_rate=2)
    x = residual_block(x, dilation_rate=4)
    x = residual_block(x, dilation_rate=8)

    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(32, activation='relu', kernel_regularizer=regularizers.l2(CONFIG['L2_REG']))(x)
    x = layers.Dense(16, activation='relu', kernel_regularizer=regularizers.l2(CONFIG['L2_REG']))(x)

    outputs = layers.Dense(2, activation='softmax', dtype='float32')(x)


    opt = optimizers.Adam(learning_rate=CONFIG['LEARNING_RATE'], clipnorm=1.0)

    model = models.Model(inputs=inputs, outputs=outputs, name="TCN_RFE")
    model.compile(optimizer=opt, loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model


current_features = list(top_35_features)
X_rfe_current = X_rf_selected.values
X_rfe_reshaped = X_rfe_current.reshape(X_rfe_current.shape[0], X_rfe_current.shape[1], 1)

while len(current_features) > CONFIG['TARGET_FEATURES']:
    print(f"RFE Iteration: {len(current_features)} features remaining...")

    idx = np.random.choice(len(X_rfe_reshaped), size=min(10000, len(X_rfe_reshaped)), replace=False)

    model = build_rfe_tcn(len(current_features))
    model.fit(X_rfe_reshaped[idx], y[idx], epochs=2, batch_size=128, verbose=0)

    inputs = tf.convert_to_tensor(X_rfe_reshaped[idx])
    with tf.GradientTape() as tape:
        tape.watch(inputs)
        predictions = model(inputs)
        loss = predictions[:, 1]

    grads = tape.gradient(loss, inputs)
    feature_importance = np.mean(np.abs(grads.numpy()), axis=(0, 2))

    worst_feature_idx = np.argmin(feature_importance)
    removed_feature = current_features.pop(worst_feature_idx)
    X_rfe_reshaped = np.delete(X_rfe_reshaped, worst_feature_idx, axis=1)

    print(f"   Removed: {removed_feature}")

final_features = current_features
print(f"\n✅ Final 20 Features Selected: {final_features}")


X_final = X_eng[final_features].values
scaler_final = MinMaxScaler()
X_final = scaler_final.fit_transform(X_final)

X_train, X_test, y_train, y_test = train_test_split(X_final, y, test_size=0.2, random_state=CONFIG['SEED'])


🧠 Starting Stage 2.2: TCN-based RFE (Dilated Causal Convolutions)...
RFE Iteration: 35 features remaining...


I0000 00:00:1768966074.344282      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15513 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0
I0000 00:00:1768966085.335456     137 service.cc:152] XLA service 0x7f5d4c024090 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1768966085.335495     137 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1768966087.982083     137 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1768966098.646756     137 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


   Removed: Fwd Act Data Packets
RFE Iteration: 34 features remaining...
   Removed: Init Bwd Win Bytes
RFE Iteration: 33 features remaining...
   Removed: Fwd IAT Max
RFE Iteration: 32 features remaining...
   Removed: Packet Length Max
RFE Iteration: 31 features remaining...
   Removed: Idle Max
RFE Iteration: 30 features remaining...
   Removed: Bwd Packet Length Max
RFE Iteration: 29 features remaining...
   Removed: Subflow Bwd Bytes
RFE Iteration: 28 features remaining...
   Removed: Fwd Packets/s
RFE Iteration: 27 features remaining...
   Removed: Packet Length Variance
RFE Iteration: 26 features remaining...
   Removed: Flow IAT Max
RFE Iteration: 25 features remaining...
   Removed: IAT_Entropy_Approx
RFE Iteration: 24 features remaining...
   Removed: Flow Packets/s
RFE Iteration: 23 features remaining...
   Removed: Idle Std
RFE Iteration: 22 features remaining...
   Removed: Bwd IAT Min
RFE Iteration: 21 features remaining...
   Removed: Bwd IAT Mean

✅ Final 20 Features Se

# ==========================================
# 6. STAGE 3: MINI-BATCH ENCODING
# ==========================================

In [ ]:
class NIDSEnvironment:
    def __init__(self, data, labels):
        self.data = data
        self.labels = labels
        self.current_step = 0
        self.max_steps = len(data) - 1

    def reset(self):
        self.current_step = 0
        return self.data[self.current_step]

    def step(self, action):
        ground_truth = self.labels[self.current_step]
        reward = 1.0 if action == ground_truth else -1.0
        self.current_step += 1
        done = self.current_step >= self.max_steps
        next_state = self.data[self.current_step] if not done else np.zeros_like(self.data[0])
        return next_state, reward, done

class ReplayBuffer:
    def __init__(self, capacity=100000):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        state, action, reward, next_state, done = map(np.stack, zip(*batch))
        return state, action, reward, next_state, done

    def __len__(self):
        return len(self.buffer)

print("Stage 3: RL Environment and Buffer ready.")

Stage 3: RL Environment and Buffer ready.


# ==========================================
# 7. STAGE 4: DQN MODEL (CNN+MLP+DQN)
# ==========================================

In [ ]:

def build_dqn_model(input_shape):
    """
    Stage 4 Architecture with Overfitting Prevention (L2 + ClipNorm)
    """
    inputs = layers.Input(shape=(input_shape, 1))


    x = layers.Conv1D(16, 3, activation='relu', padding='same',
                      kernel_regularizer=regularizers.l2(CONFIG['L2_REG']))(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Conv1D(32, 3, activation='relu', padding='same',
                      kernel_regularizer=regularizers.l2(CONFIG['L2_REG']))(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalAveragePooling1D()(x)


    x = layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(CONFIG['L2_REG']))(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(CONFIG['L2_REG']))(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(CONFIG['L2_REG']))(x)


    x = layers.Dense(32, activation='relu', kernel_regularizer=regularizers.l2(CONFIG['L2_REG']))(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Dense(16, activation='relu', kernel_regularizer=regularizers.l2(CONFIG['L2_REG']))(x)
    x = layers.Dropout(0.2)(x)

    outputs = layers.Dense(2, activation='linear', dtype='float32')(x)


    opt = optimizers.Adam(learning_rate=CONFIG['LEARNING_RATE'], clipnorm=1.0)

    model = models.Model(inputs=inputs, outputs=outputs, name="ID_RDRL_TCN_DQN")
    model.compile(loss=losses.Huber(), optimizer=opt)
    return model


dqn_model = build_dqn_model(CONFIG['TARGET_FEATURES'])
target_model = build_dqn_model(CONFIG['TARGET_FEATURES'])
target_model.set_weights(dqn_model.get_weights())

buffer = ReplayBuffer(capacity=50000)
env = NIDSEnvironment(X_train, y_train)


epsilon = 1.0
print(f"\n🎮 Starting Training on P100...")


TRAIN_FREQ = 10


best_reward = -float('inf')
patience_counter = 0

for episode in range(CONFIG['EPOCHS']):
    state = env.reset()
    state = np.reshape(state, [1, CONFIG['TARGET_FEATURES'], 1])
    total_reward = 0
    done = False


    steps = 0
    MAX_STEPS_PER_EP = 2500

    while not done and steps < MAX_STEPS_PER_EP:
        if np.random.rand() <= epsilon:
            action = np.random.randint(2)
        else:
            q_values = dqn_model.predict(state, verbose=0)
            action = np.argmax(q_values[0])

        next_state_raw, reward, done = env.step(action)
        next_state = np.reshape(next_state_raw, [1, CONFIG['TARGET_FEATURES'], 1])

        buffer.push(state, action, reward, next_state, done)
        state = next_state
        total_reward += reward
        steps += 1


        if len(buffer) > CONFIG['BATCH_SIZE'] and steps % TRAIN_FREQ == 0:
            states, actions, rewards, next_states, dones = buffer.sample(CONFIG['BATCH_SIZE'])

            states = states.reshape(-1, CONFIG['TARGET_FEATURES'], 1)
            next_states = next_states.reshape(-1, CONFIG['TARGET_FEATURES'], 1)

            target_q = target_model.predict(next_states, verbose=0)
            max_target_q = np.amax(target_q, axis=1)

            targets = dqn_model.predict(states, verbose=0)
            for i in range(CONFIG['BATCH_SIZE']):
                targets[i][actions[i]] = rewards[i] + CONFIG['GAMMA'] * max_target_q[i] * (1 - dones[i])

            dqn_model.fit(states, targets, epochs=1, verbose=0)

    target_model.set_weights(dqn_model.get_weights())

    if epsilon > CONFIG['MIN_EPSILON']:
        epsilon *= CONFIG['EPSILON_DECAY']


    if total_reward > best_reward:
        best_reward = total_reward
        patience_counter = 0
        dqn_model.save_weights("best_id_rdrl_tcn.weights.h5")
        print(f"Episode {episode+1}/{CONFIG['EPOCHS']} | Reward: {total_reward} (New Best!) | Epsilon: {epsilon:.4f}")
    else:
        patience_counter += 1
        print(f"Episode {episode+1}/{CONFIG['EPOCHS']} | Reward: {total_reward} | Patience: {patience_counter}/{CONFIG['PATIENCE']}")

    if patience_counter >= CONFIG['PATIENCE']:
        print(f"\n🛑 Early stopping triggered! Restoring best weights (Reward: {best_reward})...")
        dqn_model.load_weights("best_id_rdrl_tcn.weights.h5")
        break


🎮 Starting Training on P100...
Episode 1/15 | Reward: 58.0 (New Best!) | Epsilon: 0.9950
Episode 2/15 | Reward: -58.0 | Patience: 1/8
Episode 3/15 | Reward: 66.0 (New Best!) | Epsilon: 0.9851
Episode 4/15 | Reward: 34.0 | Patience: 1/8
Episode 5/15 | Reward: 46.0 | Patience: 2/8
Episode 6/15 | Reward: 26.0 | Patience: 3/8
Episode 7/15 | Reward: 58.0 | Patience: 4/8
Episode 8/15 | Reward: 132.0 (New Best!) | Epsilon: 0.9607
Episode 9/15 | Reward: 116.0 | Patience: 1/8
Episode 10/15 | Reward: 56.0 | Patience: 2/8
Episode 11/15 | Reward: 194.0 (New Best!) | Epsilon: 0.9464
Episode 12/15 | Reward: 28.0 | Patience: 1/8
Episode 13/15 | Reward: 90.0 | Patience: 2/8
Episode 14/15 | Reward: 188.0 | Patience: 3/8
Episode 15/15 | Reward: 24.0 | Patience: 4/8


# ==========================================
# 8. EVALUATION & METRICS
# ==========================================

In [ ]:
print("\n📊 Evaluating Performance...")

X_test_reshaped = X_test.reshape(-1, CONFIG['TARGET_FEATURES'], 1)
q_values = dqn_model.predict(X_test_reshaped)
y_pred = np.argmax(q_values, axis=1)

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"\n╔══════════════════════════════════════╗")
print(f"║    RESULTS     ")
print(f"╠══════════════════════════════════════╣")
print(f"║ Accuracy  : {acc*100:.2f}%               ║")
print(f"║ Precision : {prec*100:.2f}%               ║")
print(f"║ Recall    : {rec*100:.2f}%               ║")
print(f"║ F1-Score  : {f1*100:.2f}%               ║")
print(f"╚══════════════════════════════════════╝")


📊 Evaluating Performance...
270/270 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step

╔══════════════════════════════════════╗
║    RESULTS     
╠══════════════════════════════════════╣
║ Accuracy  : 96.73%               ║
║ Precision : 99.75%               ║
║ Recall    : 96.05%               ║
║ F1-Score  : 97.87%               ║
╚══════════════════════════════════════╝
